In [ ]:
!pip install adabound

In [ ]:
import torch
import torch.nn.functional as F
import torch.nn as nn
# import torch.optim as optim
import torch_optimizer as optim
import torch.nn.init as init
import adabound
import matplotlib.pyplot as plt
from torch.utils.data import Dataset
from torch.utils.data import DataLoader
import pandas as pd
import numpy as np
import math

from tqdm import tqdm
from tqdm import trange


In [ ]:
df = pd.read_csv('https://raw.githubusercontent.com/zorocrit/Control_Nuclear_Spins/master/initialize13Cspin/testdata_second.csv?token=GHSAT0AAAAAAB5HMDTBEBFM4V3GOPFRCIKIY7YLKSA')
# df

In [ ]:
y = df[['x', 'N', 'z']]
# y

# y = df[['Xtau', 'XN', 'Ztau', 'ZN']]
# y


In [ ]:
X = df[['Al', 'Ap']]
# X

In [ ]:
Xdata = list(np.array(X.values.tolist()))
# print(Xdata)

In [ ]:
ydata = list(np.array(y.values.tolist()))
# print(ydata)

In [ ]:
torch.manual_seed(1)

In [ ]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'

# for reproducibility
torch.manual_seed(777)
if device == 'cuda':
    torch.cuda.manual_seed_all(777)

In [ ]:
x = torch.FloatTensor(Xdata).to(device)
y = torch.FloatTensor(ydata).to(device)

In [ ]:
num_data = 2048

num_epoch = 150000

In [ ]:
model = nn.Sequential(

    nn.Linear(2,10),

    nn.ReLU(),

    nn.Linear(10,40),

    nn.ReLU(),

    nn.Linear(40,55),

    nn.ReLU(),

    nn.Linear(55,70),

    nn.ReLU(),

    nn.Linear(70,100),

    nn.ReLU(),

    nn.Linear(100,70),

    nn.ReLU(),

    nn.Linear(70,55),

    nn.ReLU(),

    nn.Linear(55,40),

    nn.ReLU(),

    nn.Linear(40,10),

    nn.ReLU(),

    nn.Linear(10,3)

).to(device)

#ReLU라는 Activation Function을 사용하여, 4개의 Linear Layer로 모델 구현

# Input layer에 1개씩 데이터가 들어가므로 nn.Linear(1,2)이며, 최종적으로 1개의 값이 나와야하기에 Output Layer는 nn.Linear(4,1)

 

loss_func = nn.L1Loss().to(device)

# optimizer = optim.SGD(model.parameters(),lr = 0.0002)

optimizer = adabound.AdaBound(model.parameters(), lr=1e-5, final_lr=0.1)

loss_array = []


for i in tqdm(range(num_epoch)) : 

    optimizer.zero_grad()

    output = model(x)

    loss = loss_func(output,y)

    loss.backward()

    optimizer.step()

 

    loss_array.append(loss)

    if(i%1000 == 0):
      print("Case "+ str(i) + ", Loss: " + str(loss.data))
    
    if i == num_epoch - 1:

        print(loss.data)

        param_list = list(model.parameters())

        #최종 학습된 마지막 결과물의 Parameter 저장

        print(param_list)

loss_array = torch.tensor(loss_array)

loss_array.detach().numpy()

plt.plot(loss_array)

plt.show()

#Loss(y_predicted - y_real)값이 어떻게 변하는지 그래프로 도식화

In [ ]:
# 임의의 입력 [73, 80, 75]를 선언
new_var =  torch.FloatTensor([[4.493041, 0.5427]]).to(device)
# 입력한 값 [73, 80, 75]에 대해서 예측값 y를 리턴받아서 pred_y에 저장
pred_y = model(new_var) 
print("훈련 후 예측값 :", pred_y) 